# Condition node: branch the flow

A question comes in, and the flow decides how to answer it:

```
question → Classify_question ("AWS" or "OTHER")
         → Route_by_topic (Condition)
             ├─ IsAWS   → Search_AWS_docs (Lambda: AWS Knowledge MCP Server)
             │            → Answer_as_AWS_expert (prompt, cites source URLs) → FlowOutput_AWS
             └─ default → Answer_general (prompt) → FlowOutput_General
```

1. A **classifier prompt** labels the question `AWS` or `OTHER`.
2. A **Condition node** checks `conditionInput == "AWS"`.
3. If true, a **Lambda node** searches AWS documentation through the public **AWS Knowledge MCP Server**, and an **AWS expert prompt** answers using only those excerpts, with the source URLs.
4. Otherwise the **default** branch sends the question to a general prompt.

> **Teaching/Learning Tip:** a Condition node evaluates expressions in order and takes the first match, with a `default` branch as the fallback. It is how a flow decides what to do next without any glue code. The comparison is exact, so the classifier must return a clean value. This one says "Reply with exactly one word" and caps `maxTokens` at 2, because without that the model adds an explanation after the label and `== "AWS"` never matches.

> **Teaching/Learning Tip:** a prompt node cannot call tools or MCP servers. To bring outside data into a flow, use a Lambda node (or a knowledge base or agent node) and wire its output into the prompt as a variable, here `{{docs}}`.

## Setup

In [ ]:
import sys
sys.path.append("..")  # flow_utils.py lives one folder up
import flow_utils

session, region, account = flow_utils.get_session_info()  # from your current AWS login
agent = session.client("bedrock-agent")            # control plane: create and manage flows
runtime = session.client("bedrock-agent-runtime")  # data plane: invoke flows
lambda_client = session.client("lambda")
iam = session.client("iam")
model_id = flow_utils.default_model_id(region, account)
print(f"Account {account} | region {region} | model {model_id}")

role_arn = flow_utils.ensure_role(iam, region, account)

## Deploy the Lambda function

`aws_docs_lambda.py` receives the question from the Lambda node, calls the AWS Knowledge MCP Server's `aws___search_documentation` tool, and returns the top excerpts with their URLs as one string. The server is public and needs no credentials, so the function uses only the Python standard library.

> **Teaching/Learning Tip:** the flow's execution role needs `lambda:InvokeFunction` on the function. `flow_utils.ensure_role` grants it for functions whose names start with `BedrockFlowsSample`.

In [ ]:
LAMBDA_NAME = "BedrockFlowsSampleAwsDocsSearch"
lambda_arn = flow_utils.deploy_lambda(lambda_client, iam, LAMBDA_NAME, "aws_docs_lambda.py")
print("lambda_arn:", lambda_arn)

## Load the template and create the flow

Three placeholders get filled in: the classifier model, the answer model, and the Lambda function ARN.

In [ ]:
template = flow_utils.load_template("conditions_flow.json", {
    "$$PROMPT_CLASSIFY_INPUT_MODEL_ID": model_id,
    "$$PROMPT_MODEL_ID": model_id,
    "$$DOCS_LAMBDA_ARN": lambda_arn,
})

flow_id = flow_utils.create_flow(
    agent, template["name"], template["description"], template["definition"], role_arn, template.get("tags"))
version, alias_id = flow_utils.deploy_flow(agent, flow_id)
print("flow_id:", flow_id, "| version:", version, "| alias_id:", alias_id)

## Invoke with an AWS question and a general question

`with_node_names=True` returns the name of the output node that produced each answer, so you can see which branch ran.

In [ ]:
questions = [
    "How do I invoke a Bedrock flow from my application?",
    "Give me a one-sentence tip for staying focused.",
]
for question in questions:
    for node_name, answer in flow_utils.invoke_flow(runtime, flow_id, alias_id, question, with_node_names=True):
        print(f"Q: {question}\nAnswered by: {node_name}\nA: {answer}\n")

## Cleanup

Delete the alias, then the version, then the flow, then the Lambda function. An alias or version blocks deleting what it points to.
The sample IAM roles are left in place so other modules can reuse them. Delete them with
`flow_utils.delete_role(iam)` and `flow_utils.delete_lambda_role(iam)` when you are completely done.

In [ ]:
flow_utils.cleanup_flow(agent, flow_id, alias_id, version)
flow_utils.delete_lambda(lambda_client, LAMBDA_NAME)